# Sesión 07 — Lab 2: VACUUM, CLONE y Data Skipping
## Mantenimiento y Optimización de Tablas Delta en Producción

**Curso:** Databricks Data Engineer Associate  
**Runtime mínimo:** DBR 13.3 LTS  
**Plataforma:** Azure Databricks con Unity Catalog habilitado  
**Dataset:** `contratos_proveedores.csv`

### Objetivos del laboratorio
1. Crear un SHALLOW CLONE y verificar que las modificaciones no afectan la tabla original
2. Inspeccionar la salud de una tabla con DESCRIBE DETAIL y DESCRIBE HISTORY
3. Ejecutar VACUUM DRY RUN y VACUUM efectivo con retención controlada
4. Verificar el impacto de VACUUM sobre Time Travel
5. Medir la efectividad del data skipping con EXPLAIN

> **Importante:** En este lab se deshabilita temporalmente el check de retención mínima de Delta para poder ejecutar VACUUM con retención corta en el entorno de pruebas. Esto **nunca** debe hacerse en producción.

In [0]:
# Configuración — ejecutar esta celda antes de comenzar
CATALOGO      = "dbassociate"
SCHEMA_BRONZE = "bronze"
VOL_LANDING   = "/Volumes/dbassociate/default/vol_landing/sesion_07"
CSV_CNT       = f"{VOL_LANDING}/contratos_proveedores.csv"

TABLA_BRONZE  = f"{CATALOGO}.{SCHEMA_BRONZE}.contratos"
TABLA_CLON    = f"{CATALOGO}.{SCHEMA_BRONZE}.contratos_test"

try:
    dbutils.fs.ls(CSV_CNT)
    print(f"OK: {CSV_CNT}")
except Exception:
    raise FileNotFoundError(
        f"Subir contratos_proveedores.csv a {VOL_LANDING} antes de continuar."
    )

## Paso 1 — Crear tabla Bronze con historial

Realizamos múltiples operaciones para construir varias versiones en el transaction log.  
Esto es necesario para que VACUUM tenga archivos históricos que limpiar y para demostrar Time Travel.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, DateType
from decimal import Decimal
from datetime import date

schema_cnt = StructType([
    StructField("contrato_id",      StringType(),       nullable=False),
    StructField("proveedor_id",     StringType(),       nullable=True),
    StructField("proveedor_nombre", StringType(),       nullable=True),
    StructField("categoria",        StringType(),       nullable=True),
    StructField("monto_anual",      DecimalType(12, 2), nullable=True),
    StructField("moneda",           StringType(),       nullable=True),
    StructField("fecha_inicio",     DateType(),         nullable=True),
    StructField("fecha_vencimiento",DateType(),         nullable=True),
    StructField("estado",           StringType(),       nullable=True),
    StructField("responsable",      StringType(),       nullable=True),
])

spark.sql(f"DROP TABLE IF EXISTS {TABLA_BRONZE}")

df_cnt = (
    spark.read
    .option("header", "true")
    .schema(schema_cnt)
    .csv(CSV_CNT)
)

# Versión 0: carga inicial
df_cnt.write.format("delta").mode("overwrite").saveAsTable(TABLA_BRONZE)
print(f"Versión 0: {df_cnt.count()} contratos cargados")


In [0]:
%sql
select * from dbassociate.bronze.contratos

In [0]:

# Versión 1: actualizar estado de contratos vencidos
spark.sql(f"""
    UPDATE {TABLA_BRONZE}
    SET estado = 'Revisado'
    WHERE estado = 'Vencido'
    AND fecha_vencimiento < '2023-01-01'
""")
print("Versión 1: contratos antiguos marcados como Revisados")

# Versión 2: cancelar contratos con estado Cancelado en USD
spark.sql(f"""
    UPDATE {TABLA_BRONZE}
    SET estado = 'Archivado'
    WHERE estado = 'Cancelado'
""")
print("Versión 2: contratos cancelados archivados")

# Versión 3: nuevo lote de contratos
df_nuevos = spark.createDataFrame([
    ("CNT-061", "PRV-161", "Nuevo Proveedor TI", "TI", Decimal("950000.00"), "MXN",
     date(2024, 7, 1), date(2025, 6, 30), "Vigente", "García López"),
    ("CNT-062", "PRV-162", "Consultor Senior", "Consultoría", Decimal("1400000.00"), "USD",
     date(2024, 7, 15), date(2025, 7, 14), "Vigente", "Hernández Cruz"),
], schema_cnt)
df_nuevos.write.format("delta").mode("append").saveAsTable(TABLA_BRONZE)
print("Versión 3: 2 contratos nuevos agregados")

print(f"\nTotal actual en {TABLA_BRONZE}: {spark.table(TABLA_BRONZE).count()} contratos")

In [0]:
%sql
select * from  dbassociate.bronze.contratos
where contrato_id = 'CNT-062'

## Paso 2 — SHALLOW CLONE: clonar sin copiar datos

SHALLOW CLONE crea una nueva tabla con su propio transaction log, pero los archivos Parquet son compartidos con la tabla original.  
Al modificar el clon, Delta escribe **nuevos archivos** para el clon sin tocar los archivos originales (copy-on-write).

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLA_CLON}")

spark.sql(f"""
    CREATE TABLE {TABLA_CLON}
    SHALLOW CLONE {TABLA_BRONZE}
""")

print(f"Clon creado: {TABLA_CLON}")
print(f"Registros en el clon: {spark.table(TABLA_CLON).count()}")
print(f"Registros en la original: {spark.table(TABLA_BRONZE).count()}")

In [0]:
%sql
select * from dbassociate.bronze.contratos_test

In [0]:
# Modificar el clon: DELETE agresivo (no haríamos esto en producción sin probar)
filas_antes_clon     = spark.table(TABLA_CLON).count()
filas_antes_original = spark.table(TABLA_BRONZE).count()

spark.sql(f"""
    DELETE FROM {TABLA_CLON}
    WHERE categoria = 'Servicios Generales'
""")

filas_despues_clon     = spark.table(TABLA_CLON).count()
filas_despues_original = spark.table(TABLA_BRONZE).count()

print(f"Clon — antes: {filas_antes_clon}, después del DELETE: {filas_despues_clon}")
print(f"Original — antes: {filas_antes_original}, después del DELETE en el clon: {filas_despues_original}")
print()
print("La tabla original NO se modificó. El DELETE escribió nuevos archivos solo en el clon (copy-on-write).")

In [0]:
%sql
select * from dbassociate.bronze.contratos_test

## Paso 2B — DEEP CLONE: copia física completa

DEEP CLONE copia los archivos Parquet a una nueva ubicación en ADLS — el clon es completamente independiente del ciclo de vida de la tabla original.

**Cuándo usar DEEP CLONE:**
- Backup real antes de una migración o transformación destructiva
- Entornos de QA completamente aislados
- Cuando se necesita garantizar que VACUUM sobre la tabla original no afecte al clon

**Contraste clave:** DEEP CLONE es más costoso (copia todos los archivos), pero garantiza independencia total. SHALLOW CLONE es instantáneo pero comparte archivos físicos con el original.

In [0]:
TABLA_DEEP = f"{CATALOGO}.{SCHEMA_BRONZE}.contratos_deep"

spark.sql(f"DROP TABLE IF EXISTS {TABLA_DEEP}")

spark.sql(f"""
    CREATE TABLE {TABLA_DEEP}
    DEEP CLONE {TABLA_BRONZE}
""")

print(f"DEEP CLONE creado: {TABLA_DEEP}")
print(f"Registros en DEEP CLONE: {spark.table(TABLA_DEEP).count()}")
print(f"Registros en original:   {spark.table(TABLA_BRONZE).count()}")

# Comparar ubicaciones físicas — la diferencia clave vs SHALLOW CLONE
print("\n--- Ubicaciones físicas ---")
loc_orig    = spark.sql(f"DESCRIBE DETAIL {TABLA_BRONZE}").collect()[0]['location']
loc_shallow = spark.sql(f"DESCRIBE DETAIL {TABLA_CLON}").collect()[0]['location']
loc_deep    = spark.sql(f"DESCRIBE DETAIL {TABLA_DEEP}").collect()[0]['location']
print(f"Original:      {loc_orig}")
print(f"SHALLOW CLONE: {loc_shallow}")
print(f"DEEP CLONE:    {loc_deep}")
print()
print("SHALLOW CLONE apunta a la misma ruta base que el original (archivos compartidos).")
print("DEEP CLONE tiene su propia ruta — archivos Parquet copiados a ADLS independientemente.")

### Comparativa: SHALLOW CLONE vs DEEP CLONE

| Aspecto | SHALLOW CLONE | DEEP CLONE |
|---|---|---|
| Archivos Parquet | Compartidos con el original | Copiados a nueva ubicación |
| Costo de creación | Instantáneo (solo metadatos) | Proporcional al tamaño de la tabla |
| Independencia vs VACUUM del original | **NO** — puede romperse | **SÍ** — completamente independiente |
| Modificaciones del clon | Copy-on-write (nuevos archivos solo para el clon) | Independiente desde el inicio |
| Caso de uso | Pruebas rápidas, desarrollo | Backup real, QA aislado, migraciones |

> **Punto clave:** La diferencia de independencia queda demostrada en el Paso 6 — después del VACUUM del Paso 5, el SHALLOW CLONE pierde el Time Travel mientras el DEEP CLONE lo conserva.

## Paso 3 — Inspeccionar la salud de la tabla

`DESCRIBE DETAIL` devuelve métricas sobre el estado físico actual de la tabla.  
`DESCRIBE HISTORY` muestra el log de operaciones con métricas de cada commit.

In [0]:
print("=== DESCRIBE DETAIL ===")
display(spark.sql(f"DESCRIBE DETAIL {TABLA_BRONZE}").select(
    "format", "numFiles", "sizeInBytes", "partitionColumns", "location"
))

In [0]:
print("=== DESCRIBE HISTORY ===")
display(spark.sql(f"DESCRIBE HISTORY {TABLA_BRONZE}").select(
    "version", "timestamp", "operation", "operationParameters", "operationMetrics"
))

## Paso 4 — VACUUM DRY RUN: ver sin eliminar

DRY RUN lista los archivos que serían eliminados con la retención especificada.  
Siempre ejecutar DRY RUN primero en producción antes de VACUUM efectivo.

In [0]:
spark.conf.set('spark.databricks.delta.retentionDurationCheck.enabled','false')

In [0]:
print("Archivos que serían eliminados con VACUUM RETAIN 168 HOURS (7 días):")
display(spark.sql(f"VACUUM {TABLA_BRONZE} RETAIN 0 HOURS DRY RUN"))

## Paso 5 — VACUUM efectivo

Para demostrar el efecto de VACUUM en el lab, deshabilitamos el check de retención mínima y usamos 0 horas.  

> **ANTIPATRÓN en producción:** `retentionDurationCheck.enabled = false` + `RETAIN 0 HOURS` puede eliminar archivos en uso por jobs concurrentes. Solo para entornos de desarrollo.

In [0]:
# ANTIPATRÓN — solo para demostración en lab
# En producción: RETAIN 168 HOURS como mínimo y con DRY RUN previo
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

spark.sql(f"VACUUM {TABLA_BRONZE} RETAIN 0 HOURS")

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

print("VACUUM completado. Archivos históricos eliminados.")
print("El check de retención fue rehabilitado después del VACUUM.")

# Verificar estado actual
display(spark.sql(f"DESCRIBE DETAIL {TABLA_BRONZE}").select("numFiles", "sizeInBytes"))

## Paso 6 — Time Travel después de VACUUM: SHALLOW vs DEEP CLONE

Después del VACUUM del Paso 5, los archivos Parquet históricos de la tabla original fueron eliminados.

**Experimento en tres tablas:**
1. **Tabla original** — fallará (`FileNotFoundException`) porque sus archivos V0 fueron eliminados por VACUUM
2. **SHALLOW CLONE** — también fallará: compartía los mismos archivos físicos que el original
3. **DEEP CLONE** — seguirá funcionando: sus archivos son completamente independientes

In [0]:
TABLA_BRONZE

In [0]:
%sql
OPTIMIZE dbassociate.bronze.contratos

In [0]:
%sql
describe history  dbassociate.bronze.contratos

In [0]:
spark.read.format("delta").option("versionAsOf", 0).table(TABLA_CLON).display()

In [0]:
print(f"Versión actual de la tabla original: {spark.table(TABLA_BRONZE).count()} registros\n")

# 1. Tabla original — falla después del VACUUM
print("--- Tabla original: VERSION AS OF 0 ---")
try:
    df_v0 = spark.read.format("delta").option("versionAsOf", 1).table(TABLA_BRONZE)
    print(f"Versión 0: {df_v0.count()} registros")
except Exception as e:
    print(f"ERROR esperado: {type(e).__name__}")
    print(f"  {str(e)[:200]}")

# 2. SHALLOW CLONE — también falla (compartía los archivos eliminados)
print()
print("--- SHALLOW CLONE: VERSION AS OF 0 ---")
try:
    df_shallow_v0 = spark.read.format("delta").option("versionAsOf", 0).table(TABLA_CLON)
    print(f"SHALLOW V0: {df_shallow_v0.count()} registros")
except Exception as e:
    print(f"ERROR esperado: {type(e).__name__} — archivos compartidos eliminados por VACUUM del original")

# 3. DEEP CLONE — sigue funcionando (archivos propios en ubicación independiente)
print()
print("--- DEEP CLONE: VERSION AS OF 0 ---")
try:
    df_deep_v0 = spark.read.format("delta").option("versionAsOf", 0).table(TABLA_DEEP)
    print(f"DEEP CLONE V0: {df_deep_v0.count()} registros — Time Travel intacto")
except Exception as e:
    print(f"ERROR inesperado en DEEP CLONE: {e}")

print()
print("Conclusión:")
print("  SHALLOW CLONE: vulnerable al VACUUM de la tabla original.")
print("  DEEP CLONE:    completamente inmune — los archivos físicos son independientes.")
print()
print("Regla en producción: VACUUM mínimo 168 horas; usar DRY RUN previo; programar fuera del horario de pipelines.")

## Paso 7 — Data Skipping con OPTIMIZE y Z-ORDER

Delta mantiene estadísticas `min/max/null_count` por archivo. Con Z-ORDER, los valores similares quedan en los mismos archivos, maximizando cuántos archivos puede saltar Spark con un filtro WHERE.

In [0]:
# Ver cuántos archivos tiene la tabla antes de OPTIMIZE
detalle_antes = spark.sql(f"DESCRIBE DETAIL {TABLA_BRONZE}").collect()[0]
print(f"Archivos antes de OPTIMIZE: {detalle_antes['numFiles']}")
print(f"Tamaño total: {detalle_antes['sizeInBytes']:,} bytes")

In [0]:
# OPTIMIZE con Z-ORDER en la columna de filtro más frecuente
# Z-ORDER en 'categoria' agrupará contratos de la misma categoría en los mismos archivos
spark.sql(f"OPTIMIZE {TABLA_BRONZE} ZORDER BY (categoria)")

detalle_despues = spark.sql(f"DESCRIBE DETAIL {TABLA_BRONZE}").collect()[0]
print(f"Archivos después de OPTIMIZE + ZORDER: {detalle_despues['numFiles']}")
print(f"Tamaño total: {detalle_despues['sizeInBytes']:,} bytes")
print()

# Verificar métricas en DESCRIBE HISTORY
hist = spark.sql(f"""
    SELECT version, operation, operationMetrics
    FROM (DESCRIBE HISTORY {TABLA_BRONZE})
    WHERE operation = 'OPTIMIZE'
    LIMIT 1
""").collect()
if hist:
    print(f"Métricas del OPTIMIZE: {hist[0]['operationMetrics']}")

In [0]:
# Verificar data skipping con EXPLAIN
# numFilesRead < numFiles indica que Spark está saltando archivos gracias a las estadísticas min/max
print("EXPLAIN de una consulta con filtro en 'categoria' (columna Z-ORDER):")
print()

df_filtrado = spark.sql(f"""
    SELECT contrato_id, proveedor_nombre, monto_anual
    FROM {TABLA_BRONZE}
    WHERE categoria = 'TI'
    AND estado = 'Vigente'
""")

# El plan físico muestra PartitionFilters y PushedFilters
df_filtrado.explain("formatted")

print()
print("Resultado:")
display(df_filtrado.orderBy("monto_anual", ascending=False))

In [0]:
# LIMPIEZA — descomentar para restablecer el ambiente
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_BRONZE}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_CLON}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_DEEP}")
# print("Limpieza completada")